# Exploratory Colab experiment

> Cleaned archive of the original graduation-project notebook. For new leakage-aware runs, use the reusable pipeline under src/ and scripts/.


In [ ]:
# Hücre 1: Gerekli Kütüphaneler ve Drive Bağlantısı
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense, BatchNormalization
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint

from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc



In [ ]:
# Hücre 2: Sabitler ve Dosya Yolları
IMG_SIZE    = 299     # ResNet50 için 299×299
BATCH_SIZE  = 32
INITIAL_LR  = 1e-4
EPOCHS_FE   = 20      # Feature Extraction epoch
EPOCHS_FT   = 80      # Fine-Tuning epoch
CHECKPOINT_PATH = 'artifacts/resnet50_filtreli1500_best.h5'

BASE_DIR    = 'data/split'
TRAIN_DIR   = os.path.join(BASE_DIR, 'train')  # İçinde SINIF1–SINIF3 klasörleri var
TEST_DIR    = os.path.join(BASE_DIR, 'test')   # İçinde SINIF1–SINIF3 klasörleri var

print("Train dizini:", TRAIN_DIR)
print("Test dizini: ", TEST_DIR)


In [ ]:
# Hücre 3: Data Augmentation & Generator’lar
# — Eğitim + Validasyon (validation_split ile)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.15,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# — Test (sadece rescale/preprocess)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
# Hücre 4: Class Weights Hesapla
num_classes = len(train_generator.class_indices)
print(f"🔢 Sınıf sayısı: {num_classes}")

classes = train_generator.classes
cw = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)
class_weights = dict(enumerate(cw))
print("⚖️ Class Weights:", class_weights)


In [ ]:
# Hücre 4: Model Tanımı (ResNet50 Backbone + Yeni Head)
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
print("✅ ResNet50 base model yüklendi.")

# — Yeni Head katmanları
x = base_model.output
x = GlobalAveragePooling2D(name='global_avg_pool')(x)
x = BatchNormalization(name='bn_1')(x)
x = Dropout(0.5, name='dropout_1')(x)
x = Dense(256, activation='relu', kernel_regularizer=l2(1e-4), name='fc_1')(x)
x = BatchNormalization(name='bn_2')(x)
x = Dropout(0.5, name='dropout_2')(x)
predictions = Dense(train_generator.num_classes, activation='softmax', name='predictions')(x)

model = Model(inputs=base_model.input, outputs=predictions)
print("✅ Model oluşturuldu.")
model.summary()


In [ ]:
# Hücre 5: Feature Extraction (只 head eğitimi)
# — Tüm backbone katmanlarını dondur
for layer in base_model.layers:
    layer.trainable = False

# — Compile
model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# — Callbacks
fe_checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH.replace('.h5', '_fe.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)
fe_reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

# — Eğitim
history_fe = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FE,
    callbacks=[fe_checkpoint, fe_reduce_lr]
)


In [ ]:
# 1) Modeli kaydet (en son ağırlıklar + mimari)
# HDF5 formatı (legacy)
model.save('resnet50_trained.h5')

# veya native Keras formatı (önerilen)
model.save('resnet50_trained.keras')

 # 2) Eğitim geçmişini (history) kaydet
import pickle

with open('training_history.pkl', 'wb') as f:
    pickle.dump(history_fe.history, f)


In [ ]:
# Hücre 6: Fine-Tuning (Son 75 katmanı açıp ince ayar ve 100’den 130’a kadar devam)

# — Backbone’dan son 75 katmanı aç, geri kalanları dondur
for layer in base_model.layers[:-75]:
    layer.trainable = False
for layer in base_model.layers[-75:]:
    layer.trainable = True

# — Compile (daha düşük LR)
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR * 0.1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# — Callbacks
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

ft_checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH.replace('.h5', '_ft75.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)
ft_reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# — Fine-tuning: 100’den başlayıp 130’a kadar
history_ft = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=130,            # Toplam 130 epoch’a kadar
    initial_epoch=100,     # 100. epoch’dan devam et
    callbacks=[ft_checkpoint, ft_reduce_lr]
)


In [ ]:
# Hücre X: Fine‐Tuning devamı (100’den 130’a kadar, son 75 katmanı açarak)

from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# 1) Modeli yükle (en iyi ağırlıklarla kaydedilmiş)
model = load_model('artifacts/resnet50_son.h5')

# 2) İlk katmanların çoğunu dondur, son 75 katmanı eğitime aç
for layer in model.layers[:-75]:
    layer.trainable = False
for layer in model.layers[-75:]:
    layer.trainable = True

# 3) Compile (daha düşük öğrenme oranı)
INITIAL_LR = 1e-4  # önceki fine‐tune aşamanızdaki LR
model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR * 0.1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4) Callbacks tanımla
checkpoint_path = 'artifacts/resnet50_son_ft75.h5'
ft_checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)
ft_reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-8,
    verbose=1
)

# 5) Fine‐tuning: 100’den 130’a kadar
history_ft = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=130,            # toplam epoch sayısı
    initial_epoch=100,     # eğitim 100’de kesilmişti
    callbacks=[ft_checkpoint, ft_reduce_lr]
)


In [ ]:
# Hücre X+1: Fine‐Tuning devamı (130’dan 150’ye, son 75 katman açık)
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# 1) Modeli yükle (en iyi ağırlıklarla kaydedilmiş)
model = load_model('artifacts/resnet50_son_ft75.h5')

# 2) İlk katmanların çoğunu dondur, son 75 katmanı eğitime aç
for layer in model.layers[:-75]:
    layer.trainable = False
for layer in model.layers[-75:]:
    layer.trainable = True

# 3) Compile (daha düşük öğrenme oranı)
NEW_LR = 1e-5  # isterseniz ince ayar için daha da küçültebilirsiniz
model.compile(
    optimizer=Adam(learning_rate=NEW_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4) Callbacks tanımla
checkpoint_path = 'artifacts/resnet50_son_ft75_150.h5'
ft_checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)
ft_reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-8,
    verbose=1
)

# 5) Fine‐tuning: 130’dan 150’ye kadar
history_ft = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=150,            # istediğimiz son epoch
    initial_epoch=130,     # eğitimin devam edeceği başlangıç epoch'u
    callbacks=[ft_checkpoint, ft_reduce_lr]
)


In [ ]:
from tensorflow.keras.models import load_model

# 1) ModelCheckpoint’in kaydettiği en iyi ağırlıkları yükleyin
best_ckpt = 'artifacts/resnet50_filtreli1500_best_ft75.h5'
model = load_model(best_ckpt)

# 2) Tüm modeli (mimari + ağırlık) tek bir .h5 dosyasında saklayın
final_path = 'artifacts/resnet50_eniyi.h5'
model.save(final_path)
print(f"✅ En iyi model tek dosyada kaydedildi:\n  {final_path}")



In [ ]:
from tensorflow.keras.models import load_model

# 1) Kaydedilmiş en iyi modeli yükle
final_path = 'artifacts/resnet50_eniyi.h5'
model = load_model(final_path)
print(f"✅ Model geri yüklendi: {final_path}\n")

# 2) Model mimarisini ve özetini incele
model.summary()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1) Tahmin ve gerçek değerler
y_true = test_generator.classes
y_pred = np.argmax(model.predict(test_generator, verbose=1), axis=1)

# 2) Confusion matrix’i oluştur
cm = confusion_matrix(y_true, y_pred)
labels = list(test_generator.class_indices.keys())

# 3) Isı haritası olarak çiz ve hücrelere sayı yaz
plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (Sayılar Gösteriliyor)')
plt.show()


In [ ]:
# Hücre 7: Eğitim ve Validasyon Sonuçları Görselleştirme
def plot_history(hist, title):
    plt.figure(figsize=(8,4))
    plt.plot(hist.history['loss'], label='train_loss')
    plt.plot(hist.history['val_loss'], label='val_loss')
    plt.title(f'{title} - Loss')
    plt.legend()
    plt.show()

    plt.figure(figsize=(8,4))
    plt.plot(hist.history['accuracy'], label='train_acc')
    plt.plot(hist.history['val_accuracy'], label='val_acc')
    plt.title(f'{title} - Accuracy')
    plt.legend()
    plt.show()

# Feature Extraction sonuçları
plot_history(history_fe, 'Feature Extraction')

# Fine-Tuning sonuçları
plot_history(history_ft, 'Fine-Tuning')


In [ ]:
# 1) Gerekli kütüphaneler
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 2) En iyi ağırlıkları yükleyelim
best_model = load_model('artifacts/resnet50_filtreli1500_best_ft75.h5')

# 3) Test generator’ı başa sar ve tahmin al
test_generator.reset()
y_prob = best_model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_generator.classes

# 4) Confusion matrix’i hesapla
cm = confusion_matrix(y_true, y_pred)
labels = list(test_generator.class_indices.keys())

# 5) Isı haritası olarak görselleştir
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Tahmin')
plt.ylabel('Gerçek')
plt.title('Test Confusion Matrix (En İyi Ağırlıklar)')
plt.show()


In [ ]:
model.save('artifacts/resnet50_eniyi.h5')


In [ ]:
# Hücre 9: ROC Eğrileri
import tensorflow as tf
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# y_true ve y_prob (ya da y_pred_probs) daha önce hesaplanmış olmalı:
# y_prob = model.predict(test_generator); y_true = test_generator.classes

# One-hot etiketler
y_true_ohe   = tf.keras.utils.to_categorical(y_true, num_classes=len(labels))
y_pred_probs = y_prob  # network’ten dönen olasılık matrisiniz

# ROC / AUC her sınıf için
plt.figure(figsize=(8,6))
for idx, label in enumerate(labels):
    fpr, tpr, _ = roc_curve(y_true_ohe[:,idx], y_pred_probs[:,idx])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f'{label} (AUC = {roc_auc:.2f})')

plt.plot([0,1], [0,1], 'k--')
plt.title('ROC Eğrileri')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

# ----- 0. Sınıf isimlerini (target_names) oluştur -----
# test_generator.class_indices: {'tip1': 0, 'tip2': 1, 'tip3': 2, ...} gibi bir sözlük döner.
# Biz bu sözlüğü indeks sırasına göre sıralayıp bir listeye koyacağız:
#
# Örneğin:
#   test_generator.class_indices = {'tip2': 1, 'tip1': 0, 'tip3': 2}
#   sorted(test_generator.class_indices.items(), key=lambda x: x[1])
#   → [('tip1', 0), ('tip2', 1), ('tip3', 2)]
#
# Böylece indeks sırasına göre isimleri elde etmiş oluyoruz.

target_names = [class_name for class_name, _ in
                sorted(test_generator.class_indices.items(), key=lambda x: x[1])]

# ----- 1. Rastgele bir test resmi seç -----
file_paths = test_generator.filepaths
random_index = random.randint(0, len(file_paths) - 1)
img_path = file_paths[random_index]

# ----- 2. Görüntüyü yükle ve normalize et -----
# IMG_SIZE: Daha önce tanımlanmış olmalı (örneğin 224, 299 vb.).
# Eğer IMG_SIZE kullanılmıyorsa, IMG_HEIGHT ve IMG_WIDTH kullanacak şekilde ayarla.
img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# ----- 3. Model ile tahmin yap ve sınıf indekslerini al -----
pred_probs = model.predict(img_array)[0]        # Örneğin: [0.10, 0.70, 0.20]
predicted_idx = np.argmax(pred_probs)
predicted_class = target_names[predicted_idx]

true_idx = test_generator.classes[random_index]
true_class = target_names[true_idx]

# ----- 4. Sonuçları ekrana bastır -----
print(f"Gerçek:   {true_class}")
print(f"Tahmin:   {predicted_class}")
print()  # boş bir satır
for i, name in enumerate(target_names):
    print(f"{name}: {pred_probs[i]:.2f}")

# ----- 5. Alt kısımda yalnızca resmi göster -----
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.show()